# Grouped TSAM Workflow

This notebook is the interactive client for the grouped Option 1 workflow. The reusable loading, validation, calendar grouping, TSAM aggregation, CSV schema construction, and chart builders live in `tsam_workflows`.


## Table of Contents

- [Method Overview](#Method-Overview)
- [Normalization](#Normalization)
- [Feature Preservation](#Feature-Preservation)
  - [Preservation Objectives](#Preservation-Objectives)
  - [Clustering Influence](#Clustering-Influence)
  - [Current Baseline](#Current-Baseline)
- [Imports And Configuration](#Imports-And-Configuration)
- [Run Workflow](#Run-Workflow)
- [Output Tables](#Output-Tables)
- [Summary Charts](#Summary-Charts)
- [Group-Level TSAM Diagnostic Drilldowns](#Group-Level-TSAM-Diagnostic-Drilldowns)
- [Optional CSV Export](#Optional-CSV-Export)


## Method Overview

The workflow clusters days separately by calendar month and by working/non-working status. With the default 5 working-day and 2 non-working-day representative configuration, the year is split into 24 groups and produces 84 representative days.


# Normalization

TSAM handles basic per-column normalization internally. If normalized columns or feature groups should contribute differently to clustering, their distance contributions can be normalized in a second step. TSAM implements this additional normalization through the top-level `weights=` argument.

**Current grouped-workflow support**

The grouped workflow and CLI use TSAM's default per-column normalization with equal column weights. Additional normalization through `normalize_column_means` or `weights` is not supported by the grouped workflow or CLI. The methods below are retained as technical reference and require custom code outside the current interface.

The clustering pipeline is:

`raw columns -> per-column min-max scaling -> optional mean scaling -> optional contribution normalization through weights -> clustering`

**Built-in normalization**

- TSAM independently maps each input column's observed range to `0..1` before clustering.
- This transformation is internal; representative values are returned in their original units.
- `ClusterConfig(normalize_column_means=True)` optionally divides each normalized column by its mean. It is disabled in the baseline.
- External affine scaling, such as min-max or z-score normalization, is generally redundant because TSAM subsequently applies its own min-max scaling.

**Why additional contribution normalization may be needed**

Equal column ranges do not guarantee equal influence. Distance still sums squared differences across every column and timestep, so a feature group's contribution depends on both its column count and normalized variability. `tsam.aggregate(weights=...)` is the implementation mechanism for normalizing these contributions after min-max scaling.

**Candidate additional normalization strategies**

The baseline uses no additional contribution normalization. The alternatives below are implemented through column weights:

| Strategy | Purpose |
|---|---|
| No additional normalization | Keep TSAM's default equal column weights |
| Feature-group contribution normalization | Use `1 / sqrt(group_column_count)` to diagnose and correct column-count imbalance |
| Physical-scale normalization | Where defensible conversions exist, scale normalized differences to common physical units using demand range for load and installed capacity multiplied by observed capacity-factor range for renewable availability |

**Illustrative pseudocode**

The TSAM-level pseudocode below illustrates how these strategies can be expressed without modifying the raw input data. It is intentionally non-executable and cannot be selected through the current CLI.

```python
NORMALIZATION_MODE = "none"
# Alternatives: "equal_feature_groups", "physical_scale"


def build_normalization_weights(data, mode):
    if mode == "none":
        return None

    columns_by_group = group_columns_by_feature(data.columns)

    if mode == "equal_feature_groups":
        return {
            column: 1 / sqrt(len(group_columns))
            for group_columns in columns_by_group.values()
            for column in group_columns
        }

    if mode == "physical_scale":
        return {
            column: physical_range_in_common_power_units(column)
            for column in data.columns
        }

    raise ValueError("Unknown normalization mode")


normalization_weights = build_normalization_weights(
    group_features, NORMALIZATION_MODE
)

aggregation_result = tsam.aggregate(
    data=group_features,
    n_clusters=n_clusters,
    weights=normalization_weights,
    cluster=tsam.ClusterConfig(
        method="hierarchical",
        representation="medoid",
        normalize_column_means=False,
    ),
    preserve_column_means=False,  # Feature preservation, not normalization.
)
```

For equal feature-group contributions, `group_column_count * (1 / sqrt(group_column_count)) ** 2 = 1`. The `physical_range_in_common_power_units` helper remains abstract until the required installed-capacity and unit-conversion metadata is available.

No additional normalization strategy is selected in the baseline. Alternatives should be compared using group-level normalized errors, peak and extreme preservation, distance contributions, and downstream model results.

**Mean preservation is separate**

`preserve_column_means` is post-clustering behavior, not normalization. The baseline keeps it `False` so medoid profiles remain untouched observed days. Setting it to `True` rescales TSAM's representative outputs to preserve weighted means, so they are no longer exact medoids. The notebook's separately sliced reduced rows would remain original unless they were instead built from those rescaled representatives.

[TSAM API reference](https://tsam.readthedocs.io/en/latest/api/tsam/api/)


# Feature Preservation

Temporal aggregation cannot retain every original value. Feature preservation therefore starts by choosing which property should be retained in the representative periods or reconstructed series.

**Current grouped-workflow support**

The grouped workflow and CLI currently use hierarchical clustering with medoid representation. Selective mean preservation, extreme-period preservation, and distribution representations are not supported by the grouped workflow or CLI. The alternatives below are retained as technical reference and require custom code outside the current interface.

## Preservation Objectives

| Objective | TSAM setting | What it preserves |
|---|---|---|
| Exact observed profiles | `ClusterConfig(representation="medoid")` | Complete observed periods, including relationships between columns |
| Annual means and totals | `preserve_column_means=True` | Occurrence-weighted column means; totals follow when the represented duration is unchanged |
| Extremes | `ExtremeConfig(...)` | Configured minimum or maximum values or periods for selected columns |
| Duration-curve shape | `ClusterConfig(representation=tsam.Distribution(...))` | An approximation of each column's value distribution, optionally including its minimum and maximum |

Medoid and distribution representations apply to every input column. To preserve means only for selected columns, enable `preserve_column_means` and place every non-selected column in `rescale_exclude_columns`. `ExtremeConfig` can target selected columns directly.

**Illustrative pseudocode**

Each preservation objective changes different TSAM arguments. The TSAM-level example below is intentionally non-executable and cannot be selected through the current CLI.

```python
PRESERVATION_MODE = "exact_profiles"
# Alternatives: "selected_means", "selected_extremes", "all_distributions"

PRESERVED_FEATURE_GROUPS = {"demand"}


def build_preservation_options(columns, mode, feature_groups):
    selected_columns = columns_for_feature_groups(columns, feature_groups)
    non_selected_columns = [
        column for column in columns if column not in selected_columns
    ]

    options = {
        "cluster": tsam.ClusterConfig(
            method="hierarchical",
            representation="medoid",
        ),
        "preserve_column_means": False,
        "rescale_exclude_columns": None,
        "extremes": None,
    }

    if mode == "exact_profiles":
        return options

    if mode == "selected_means":
        options["preserve_column_means"] = True
        options["rescale_exclude_columns"] = non_selected_columns
        return options

    if mode == "selected_extremes":
        options["extremes"] = tsam.ExtremeConfig(
            method="append",
            max_value=selected_columns,
        )
        return options

    if mode == "all_distributions":
        # Distribution representation applies to every input column.
        options["cluster"] = tsam.ClusterConfig(
            method="hierarchical",
            representation=tsam.Distribution(
                scope="global",
                preserve_minmax=True,
            ),
        )
        return options

    raise ValueError("Unknown preservation mode")


preservation_options = build_preservation_options(
    columns=group_features.columns,
    mode=PRESERVATION_MODE,
    feature_groups=PRESERVED_FEATURE_GROUPS,
)

aggregation_result = tsam.aggregate(
    data=group_features,
    n_clusters=n_clusters,
    **preservation_options,
)

# Use TSAM's final values for modes that create synthetic representatives.
representative_values = aggregation_result.cluster_representatives
```

- `selected_means` rescales only the selected feature-family columns.
- `selected_extremes` demonstrates maximum-value preservation. Minimum values or extreme period totals use the corresponding `ExtremeConfig` fields.
- `all_distributions` cannot honor `PRESERVED_FEATURE_GROUPS`; it affects every input column.
- Mean preservation must export `cluster_representatives`; otherwise the current original-row slicing restores the unscaled medoid values. Distribution representatives are synthetic and have no original row to slice, so that mode also requires a different export path.

## Clustering Influence

The top-level `weights=` argument changes how strongly columns influence clustering distances. It can improve representation of important features, but it does not guarantee preservation of their means, extremes, or distributions.

## Current Baseline

The baseline uses hierarchical clustering with medoid representation, no mean rescaling, no extreme-period configuration, and default equal column weights. It preserves the exact profiles of selected medoid days for every column, but it does not guarantee annual means, extremes, or duration curves.

**API references:** [ClusterConfig](https://tsam.readthedocs.io/en/latest/api/tsam/api/#tsam.config.ClusterConfig) | [ExtremeConfig](https://tsam.readthedocs.io/en/latest/api/tsam/api/#tsam.config.ExtremeConfig)

## Imports And Configuration

The notebook runs the original 5-working/2-non-working representative configuration for all available countries. Country drilldowns use full country names and initially select Germany when available.


In [ ]:
from pathlib import Path
from typing import Any

import ipywidgets as widgets
from IPython.display import display

from tsam_workflows.charts import (
    build_assignment_calendar_figure,
    build_group_accuracy_figure,
    build_representative_weights_figure,
    style_tsam_figure,
)
from tsam_workflows.config import (
    GroupedWorkflowConfig,
    country_options,
    default_dataset_specs,
)
from tsam_workflows.grouped import run_grouped_workflow


In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "approach_1"
NOTEBOOK_COUNTRIES = None
NOTEBOOK_WORKING_CLUSTERS = 5
NOTEBOOK_NON_WORKING_CLUSTERS = 2

config = GroupedWorkflowConfig(
    year=2025,
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    countries=NOTEBOOK_COUNTRIES,
    working_clusters=NOTEBOOK_WORKING_CLUSTERS,
    non_working_clusters=NOTEBOOK_NON_WORKING_CLUSTERS,
    cluster_method="hierarchical",
)
dataset_specs = default_dataset_specs(DATA_DIR)


## Run Workflow

`run_grouped_workflow` returns the three persisted tables, per-group TSAM results, coverage diagnostics, group accuracy, and the country/feature lookup used by the notebook widgets.


In [ ]:
result = run_grouped_workflow(config, dataset_specs)


In [ ]:
result.dataset_coverage


In [ ]:
summary = {
    "groups": len(result.tsam_results_by_group),
    "representative_days": len(result.representative_days),
    "day_assignments": len(result.day_assignments_df),
    "reduced_hourly_rows": len(result.reduced_hourly_df),
    "selected_countries": ", ".join(result.selected_countries),
}
summary


## Output Tables


In [ ]:
result.representative_days.head()


In [ ]:
result.day_assignments_df.head()


In [ ]:
result.reduced_hourly_df.head()


## Summary Charts


In [ ]:
build_assignment_calendar_figure(result).show(config={"responsive": True})


In [ ]:
build_representative_weights_figure(result).show(config={"responsive": True})


In [ ]:
build_group_accuracy_figure(result).show(config={"responsive": True})


## Group-Level TSAM Diagnostic Drilldowns

These controls are notebook-only. The CLI exports each selected widget state as a standalone offline HTML file instead of exporting live ipywidgets.


In [ ]:
GROUP_OPTIONS = list(result.group_ids)
COUNTRY_OPTIONS = country_options(
    list(result.feature_columns_by_country_and_group)
)
DEFAULT_COUNTRY = (
    "DE"
    if "DE" in result.feature_columns_by_country_and_group
    else COUNTRY_OPTIONS[0][1]
)


def feature_group_options(country: str) -> list[str]:
    return list(result.feature_columns_by_country_and_group[country])


def make_group_dropdown() -> widgets.Dropdown:
    return widgets.Dropdown(
        options=GROUP_OPTIONS,
        description="Group:",
        layout=widgets.Layout(width="360px"),
    )


def make_country_dropdown() -> widgets.Dropdown:
    return widgets.Dropdown(
        options=COUNTRY_OPTIONS,
        value=DEFAULT_COUNTRY,
        description="Country:",
        layout=widgets.Layout(width="220px"),
    )


def make_feature_group_dropdown(country: str) -> widgets.Dropdown:
    return widgets.Dropdown(
        options=feature_group_options(country),
        description="Feature:",
        layout=widgets.Layout(width="260px"),
    )


def selected_columns(country: str, feature_group: str) -> list[str]:
    return result.feature_columns_by_country_and_group[country][feature_group]


In [ ]:
def display_group_plot(plot_function: Any) -> None:
    group_dropdown = make_group_dropdown()
    output = widgets.interactive_output(plot_function, {"group_id": group_dropdown})
    display(widgets.VBox([widgets.HBox([group_dropdown]), output]))


def display_group_feature_plot(plot_function: Any) -> None:
    group_dropdown = make_group_dropdown()
    country_dropdown = make_country_dropdown()
    feature_group_dropdown = make_feature_group_dropdown(country_dropdown.value)

    def update_feature_groups(change: dict[str, Any]) -> None:
        selected = feature_group_dropdown.value
        options = feature_group_options(change["new"])
        feature_group_dropdown.options = options
        feature_group_dropdown.value = selected if selected in options else options[0]

    country_dropdown.observe(update_feature_groups, names="value")
    output = widgets.interactive_output(
        plot_function,
        {
            "group_id": group_dropdown,
            "country": country_dropdown,
            "feature_group": feature_group_dropdown,
        },
    )
    display(widgets.VBox([widgets.HBox([group_dropdown, country_dropdown, feature_group_dropdown]), output]))


In [ ]:
def show_cluster_representatives(group_id: str, country: str, feature_group: str) -> None:
    fig = result.tsam_results_by_group[group_id].plot.cluster_representatives(
        columns=selected_columns(country, feature_group),
        title="",
    )
    style_tsam_figure(fig, f"Cluster representative profiles: {group_id}").show(config={"responsive": True})


display_group_feature_plot(show_cluster_representatives)


In [ ]:
def show_cluster_members(group_id: str, country: str, feature_group: str) -> None:
    fig = result.tsam_results_by_group[group_id].plot.cluster_members(
        columns=selected_columns(country, feature_group),
        slider="cluster",
        title="",
    )
    style_tsam_figure(fig, f"Cluster members: {group_id}").show(config={"responsive": True})


display_group_feature_plot(show_cluster_members)


In [ ]:
def show_cluster_weights(group_id: str) -> None:
    fig = result.tsam_results_by_group[group_id].plot.cluster_weights(title="")
    style_tsam_figure(fig, f"Cluster weights: {group_id}").show(config={"responsive": True})


display_group_plot(show_cluster_weights)


In [ ]:
def show_cluster_accuracy(group_id: str) -> None:
    fig = result.tsam_results_by_group[group_id].plot.accuracy(title="")
    style_tsam_figure(fig, f"Cluster accuracy: {group_id}").show(config={"responsive": True})


display_group_plot(show_cluster_accuracy)


In [ ]:
def show_full_resolution_comparison(group_id: str, country: str, feature_group: str) -> None:
    fig = result.tsam_results_by_group[group_id].plot.compare(
        columns=selected_columns(country, feature_group),
        title="",
    )
    style_tsam_figure(fig, f"Original vs reconstructed: {group_id}").show(config={"responsive": True})


display_group_feature_plot(show_full_resolution_comparison)


In [ ]:
def show_cluster_residuals(group_id: str, country: str, feature_group: str) -> None:
    fig = result.tsam_results_by_group[group_id].plot.residuals(
        columns=selected_columns(country, feature_group),
        title="",
    )
    style_tsam_figure(fig, f"Residuals: {group_id}").show(config={"responsive": True})


display_group_feature_plot(show_cluster_residuals)


## Optional CSV Export

The CLI is the preferred reproducible export path because it stages outputs before publishing. Set `SAVE_OUTPUTS = True` when you want this notebook to write the same CSV filenames during an interactive run.


In [ ]:
SAVE_OUTPUTS = False

if SAVE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    (result.reduced_hourly_df.reset_index()).to_csv(
        OUTPUT_DIR / "reduced_hourly_df.csv",
        index=False,
    )
    result.representative_days.to_csv(OUTPUT_DIR / "representative_days.csv", index=False)
    (result.day_assignments_df.reset_index()).to_csv(
        OUTPUT_DIR / "day_assignments_df.csv",
        index=False,
    )
    export_summary = {
        "reduced_hourly_df.csv": len(result.reduced_hourly_df),
        "representative_days.csv": len(result.representative_days),
        "day_assignments_df.csv": len(result.day_assignments_df),
    }
else:
    export_summary = {"csv_export": "skipped", "output_dir": str(OUTPUT_DIR)}

export_summary
